In [ ]:


import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# ------------------------------------------------------------
# 1. PREPARE DATA
# ------------------------------------------------------------

fact_sales_fe=pd.read_csv(r"C:\Users\RohanS\Documents\retail project\data retail processed\fact_sales_fe.csv")
df = fact_sales_fe.copy()

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)


# ------------------------------------------------------------
# 2. CREATE REFERENCE DATE
# ------------------------------------------------------------

reference_date = (
    df["order_purchase_timestamp"].max()
    + pd.Timedelta(days=1)
)


# ------------------------------------------------------------
# 3. CALCULATE RFM
# ------------------------------------------------------------

rfm = (
    df.groupby("customer_id")
      .agg(
          Recency=(
              "order_purchase_timestamp",
              lambda x:
              (reference_date - x.max()).days
          ),

          Frequency=(
              "order_id",
              "nunique"
          ),

          Monetary=(
              "payment_value",
              "sum"
          )
      )
      .reset_index()
)


# ------------------------------------------------------------
# 4. REMOVE EXTREME OUTLIERS
# ------------------------------------------------------------

for col in ["Recency", "Frequency", "Monetary"]:

    lower = rfm[col].quantile(0.01)
    upper = rfm[col].quantile(0.99)

    rfm[col] = rfm[col].clip(
        lower,
        upper
    )




rfm_log = np.log1p(
    rfm[
        ["Recency", "Frequency", "Monetary"]
    ]
)



scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(
    rfm_log
)



kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

rfm["Cluster"] = kmeans.fit_predict(
    rfm_scaled
)


cluster_profile = (
    rfm.groupby("Cluster")
       .agg(
           Customers=("customer_id", "count"),
           Avg_Recency=("Recency", "mean"),
           Avg_Frequency=("Frequency", "mean"),
           Avg_Monetary=("Monetary", "mean")
       )
       .reset_index()
)


cluster_profile["RFM_score"] = (

    (
        cluster_profile["Avg_Frequency"]
        /
        cluster_profile["Avg_Frequency"].max()
    )

    +

    (
        cluster_profile["Avg_Monetary"]
        /
        cluster_profile["Avg_Monetary"].max()
    )

    +

    (
        1
        -
        cluster_profile["Avg_Recency"]
        /
        cluster_profile["Avg_Recency"].max()
    )

) / 3


cluster_profile = cluster_profile.sort_values(
    "RFM_score",
    ascending=False
).reset_index(drop=True)



segment_names = [
    "Champions",
    "Loyal Customers",
    "Potential Loyalists",
    "At Risk",
    "Lost Customers"
]

cluster_to_segment = {
    cluster_profile.loc[i, "Cluster"]:
    segment_names[i]
    for i in range(
        len(cluster_profile)
    )
}


rfm["Customer_Segment"] = (
    rfm["Cluster"]
    .map(cluster_to_segment)
)




customer_segments = rfm[
    [
        "customer_id",
        "Recency",
        "Frequency",
        "Monetary",
        "Cluster",
        "Customer_Segment"
    ]
].copy()


customer_segments[
    ["Recency", "Frequency", "Monetary"]
] = customer_segments[
    ["Recency", "Frequency", "Monetary"]
].round(2)



print("\nCustomer segments:")
print(
    customer_segments.head(20)
)


print("\nSegment distribution:")
print(
    customer_segments[
        "Customer_Segment"
    ].value_counts()
)


print("\nCluster profile:")
print(
    cluster_profile[
        [
            "Cluster",
            "Customers",
            "Avg_Recency",
            "Avg_Frequency",
            "Avg_Monetary",
            "RFM_score"
        ]
    ].round(2)
)




customer_segments.to_csv(
    "customer_segments_8_5.csv",
    index=False
)

cluster_profile.to_csv(
    "customer_segment_profile_8_5.csv",
    index=False
)


